# The Order Book and Order Types

When your code sends an order, it lands in a precise data structure called the order book, and the type of order you send decides how — and whether — it fills. Getting this wrong is one of the most expensive beginner errors: a careless market order in a thin book can cost more than a week of edge. This lesson makes the order book concrete and walks through every order type you'll actually use.

By the end of this lesson you will be able to:

- Read a limit order book, including bids, asks, and depth
- Distinguish market, limit, stop, and stop-limit orders and when to use each
- Explain IOC and FOK time-in-force conditions
- Define maker versus taker and why the distinction affects your costs
- Trace how a large order "walks the book" and what that does to your fill price



## 1. The Limit Order Book

The **limit order book (LOB**)** is simply the exchange's sorted list of all resting limit orders for an instrument. It has two sides:

1. **Bids** - Orders to buy , sorted from highest price(best) down.
2. **Asks** - Orders to sell , sorted from lowest price(best) up.

Each level shows a price and the total **size**(quantity) resting there. The total quantity available across levels is the books depth. Here is a small book:

```
        Price    Size
Asks    50.09    400
        50.08    250
        50.07    150     <- best ask (lowest sell)
------------------------ spread = 0.03
Bids    50.04    300     <- best bid (highest buy)
        50.03    500
        50.02    220
```

The **best bid** (50.04) and **best ask** (50.07) define the top of book. The gap between them — 0.03 here — is the **spread**. The midpoint, 50.055, is often used as a "fair" reference price, but note you can't actually trade at the mid with a simple order. Depth matters: this book can absorb 150 shares of buying at 50.07 before the price you pay rises.

The best bid (50.04) and best ask (50.07) define the top of book. The gap between them — 0.03 here — is the spread. The midpoint, 50.055, is often used as a "fair" reference price, but note you can't actually trade at the mid with a simple order. Depth matters: this book can absorb 150 shares of buying at 50.07 before the price you pay rises.

## 2. Order Types

### 1. Market Order:
A market order says "fill me immediately at the best available price, whatever it is." It guarantees execution but not price. It always crosses the spread — buying lifts the ask, selling hits the bid. In a deep book this is fine; in a thin one it can fill far from where you expected. Never assume you'll get the price on your screen.

### 2. Limit Order

A limit order says "fill me only at this price or better." A buy limit at 50.05 will not pay more than 50.05. It guarantees price but not execution — if the market never reaches your limit, you simply don't trade. A limit order priced away from the market rests in the book and adds liquidity; a limit order priced to cross executes immediately like a market order but with a price cap.

### 3. Stop Order

A stop order is dormant until the market touches a trigger price, then becomes a market order. A common use is a protective stop-loss: "if price falls to 49.50, sell at market." Stops are not visible in the book until triggered. The danger: once triggered they become market orders, so in a fast drop your fill can be well below the stop level.


### 4. Stop Limit Order

A stop-limit combines the two: when the trigger is hit, it submits a limit order rather than a market order. "If price falls to 49.50, place a sell limit at 49.40." This caps how bad your fill can be — but if price gaps straight through 49.40, the limit may never fill and you're left holding the position. You trade execution certainty for price protection.


>Rule of thumb: market and stop orders prioritise getting done; limit and stop-limit orders prioritise price. You can rarely have both at once.

A compact way to hold the four order types in your head is a 2×2 grid along two questions: does it trigger on a condition? and does it cap the price?


```
                    no price cap        price cap
not triggered:      market order        limit order
triggered:          stop order          stop-limit order

```

A market order is the "just do it" choice; a limit order adds a price ceiling/floor; a stop is a market order that waits for a trigger; a stop-limit is a limit order that waits for a trigger. Every order you place is one of these four combinations, and choosing correctly is choosing which of execution-certainty or price-certainty you're willing to give up.

## 3. Time-in-force: IOC and FOK
Beyond price, you can control how long an order lives:

- **IOC (Immediate-Or-Cancel)**. Fill whatever you can right now, cancel the rest. Useful for taking available liquidity without leaving a resting order behind.
- **FOK (Fill-Or-Kill)**. Fill the entire quantity immediately or cancel the whole thing. No partial fills. Useful when a partial position is worse than none.
- **Day / GTC**. A plain limit order usually rests until the end of the day (Day) or until you cancel it (Good-Til-Canceled).

The distinction between IOC and FOK matters more than it looks. Suppose you want 1,000 shares but only 600 are available at your price. An IOC takes the 600 and cancels the rest — you end up with a partial position. A FOK sees it can't get all 1,000 and cancels entirely — you end up with nothing. Which you want depends on the strategy: if a partial fill leaves you with awkward, unhedged risk (say, one leg of a pair trade), FOK protects you; if any fill is better than none, IOC is right. Beginners often leave time-in-force at the default and are then surprised by partial fills; choosing deliberately is part of execution discipline.

## 4. Maker Vs Taker

This distinction drives your fees and your strategy design:
- A **Maker** posts a resting limit order that adds liquidity to the book. Someone else later trades against it.
- A **Taker**  sends an order that removes liquidity by matching against a resting order (any market order, or a marketable limit).

Most exchanges charge takers a fee and often pay makers a small rebate, because makers provide the liquidity everyone relies on. For a high-frequency strategy, the difference between paying a taker fee and earning a maker rebate can be the entire difference between profit and loss. The catch: maker orders aren't guaranteed to fill, so you trade fee savings for execution uncertainty.

Make the economics concrete. Suppose an exchange charges takers 5 basis points and pays makers a 2 bp rebate. On a $50,000 trade, taking costs you 0.0005 50000 = $25, while making earns you 0.0002 50000 = $10 — a $35 swing on a single trade purely from which side of the book you're on. For a strategy trading hundreds of times a day, that swing is the whole business. This is why serious short-horizon strategies are designed around posting liquidity and patiently waiting in the queue rather than crossing the spread. The price of that rebate is real, though: your resting order might never fill, or fill only after the market has moved against the reason you wanted in.

